# MedAssist triage model: train, evaluate, export (Kaggle GPU)

Run top to bottom on Kaggle with **GPU on** and **Internet on**. It replaces the local training setup: no local dataset or GPU needed.

**What it does**
1. Regenerates the synthetic dataset reproducibly (seeded) using the repo's own generator, and records which symptom combination each sample came from.
2. Splits **by symptom combination**, so test sentences use symptom sets the model never saw in training. It also trains one model on a plain random split to show how much that split inflates accuracy.
3. Trains BioBERT, DistilBERT and PubMedBERT over several seeds, plus a TF-IDF + Logistic Regression baseline.
4. Tests on an independent text set (Symptom2Disease) if you attach it.
5. Measures calibration, selective prediction (answer only when confident), the emergency rule engine, and latency.
6. Writes `model-card.json` for the backend, and optionally pushes the best model to Hugging Face.

**Before you run**
- Optional but recommended: **Add Input** -> search `Symptom2Disease` (a Kaggle dataset of natural-language symptom descriptions) and attach it.
- To push to Hugging Face: Add-ons -> Secrets -> add `HF_TOKEN`, then set `PUSH_TO_HF = True` below.
- Set `QUICK = True` first for a smoke test (a few minutes).

All numbers come from this run. Nothing is hard-coded.

In [ ]:
%pip install -q huggingface_hub

In [ ]:
import os, sys, json, time, random, itertools, glob, shutil, subprocess, copy, warnings, datetime
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ------------------------------- settings -------------------------------------
QUICK = False                      # True = tiny smoke test
SEEDS = [0, 1, 2]                  # training seeds per model
MODELS = {                         # name -> Hugging Face model id
    "BioBERT":    "monologg/biobert_v1.1_pubmed",
    "DistilBERT": "distilbert-base-uncased",
    "PubMedBERT": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
}
N_COMBOS_PER_CLASS  = 240          # symptom combinations sampled per condition
TEMPLATES_PER_COMBO = 4            # sentence templates per combination
MAX_LEN, EPOCHS, BATCH, LR = 64, 3, 64, 3e-5
DATA_SEED = 1234

PUSH_TO_HF = False                 # True to upload the best model
HF_REPO    = "your-username/medassist-bert-v2"   # use a NEW repo name first, then set HF_MODEL_ID

REPO_URL = "https://github.com/yogeshroyal63-beep/medassit"
WORK     = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
OUT      = f"{WORK}/results"
BEST_DIR = f"{WORK}/best_model"
os.makedirs(OUT, exist_ok=True)

if QUICK:
    SEEDS = [0]
    MODELS = {"BioBERT": MODELS["BioBERT"], "DistilBERT": MODELS["DistilBERT"]}
    N_COMBOS_PER_CLASS, EPOCHS = 30, 1
print("QUICK" if QUICK else "FULL", "| seeds", SEEDS, "| models", list(MODELS))

## 1. Data: regenerate with the repo's generator, keeping the combination id

In [ ]:
REPO = f"{WORK}/medassit"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
# the generator lives in ml/ after the cleanup (ai_service/ before it); the AI package is needed for the emergency rules
for sub in ("ml", "ai_service"):
    if os.path.exists(f"{REPO}/{sub}/generate_dataset.py"):
        sys.path.insert(0, f"{REPO}/{sub}")
        break
sys.path.insert(0, f"{REPO}/ai_service")
import generate_dataset as gd
print("conditions:", len(gd.CONDITIONS), "| templates:", len(gd.TEMPLATES))

rng = random.Random(DATA_SEED)
random.seed(DATA_SEED)             # gd.apply_noise uses the global RNG
rows, seen = [], set()
for cond, syms in gd.CONDITIONS.items():
    combos = []
    for k in (2, 3, 4):
        combos += list(itertools.combinations(syms, k))
    rng.shuffle(combos)
    for ci, combo in enumerate(combos[:N_COMBOS_PER_CLASS]):
        for tpl in rng.sample(gd.TEMPLATES, min(TEMPLATES_PER_COMBO, len(gd.TEMPLATES))):
            text = gd.apply_noise(gd.fill_template(tpl, list(combo)))
            if (text, cond) in seen:
                continue
            seen.add((text, cond))
            rows.append((text, cond, f"{cond}|{ci}"))

df = pd.DataFrame(rows, columns=["text", "label", "group"])
print("samples:", len(df), "| symptom-combination groups:", df.group.nunique())
print(df.sample(3, random_state=0).to_string())

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

le = LabelEncoder().fit(df.label)
df["y"] = le.transform(df.label)
NC = len(le.classes_)

def group_split(frame, test_frac, seed):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)
    a, b = next(gss.split(frame, groups=frame.group))
    return frame.iloc[a].reset_index(drop=True), frame.iloc[b].reset_index(drop=True)

# (A) split by symptom combination: test sentences use symptom sets never seen in training
trainval, test_combo = group_split(df, 0.15, 1)
train_combo, val_combo = group_split(trainval, 0.1765, 2)
for a, b in [(train_combo, val_combo), (train_combo, test_combo), (val_combo, test_combo)]:
    assert not set(a.group) & set(b.group), "combination leaked across splits"

# (B) plain random split, like the original training script
tr_r, tmp = train_test_split(df, test_size=0.3, stratify=df.y, random_state=3)
val_r, test_r = train_test_split(tmp, test_size=0.5, stratify=tmp.y, random_state=3)
leak = test_r.group.isin(set(tr_r.group)).mean()
print(f"classes: {NC} | combo split train/val/test: {len(train_combo)}/{len(val_combo)}/{len(test_combo)}")
print(f"random split: {leak*100:.1f}% of test samples share a symptom combination with a training sample")

## 2. Metrics helpers

In [ ]:
from sklearn.metrics import f1_score

def topk_accuracy(probs, y, k=3):
    top = np.argsort(-probs, axis=1)[:, :k]
    return float(np.mean([y[i] in top[i] for i in range(len(y))]))

def ece_score(probs, y, bins=15):
    conf = probs.max(1); pred = probs.argmax(1); correct = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(total)

def selective_accuracy(probs, y, coverages=(1.0, 0.9, 0.8, 0.7)):
    conf = probs.max(1); correct = (probs.argmax(1) == y).astype(float)
    order = np.argsort(-conf); out = []
    for c in coverages:
        n = max(1, int(round(c * len(y))))
        out.append({"coverage": c, "accuracy": float(correct[order[:n]].mean())})
    return out

def evaluate_probs(probs, y):
    pred = probs.argmax(1)
    return {
        "n": int(len(y)),
        "accuracy": float((pred == y).mean()),
        "macro_f1": float(f1_score(y, pred, average="macro", labels=np.unique(y))),
        "top3": topk_accuracy(probs, y, 3),
        "ece": ece_score(probs, y),
    }

## 3. Baseline: TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def full_probs(clf, X):
    p = clf.predict_proba(X)
    out = np.zeros((p.shape[0], NC)); out[:, clf.classes_] = p
    return out

t0 = time.time()
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(train_combo.text)
clf = LogisticRegression(max_iter=300, C=5.0).fit(Xtr, train_combo.y)
base_combo = evaluate_probs(full_probs(clf, vec.transform(test_combo.text)), test_combo.y.values)
print(f"TF-IDF + LR on held-out combinations: acc {base_combo['accuracy']:.3f}, macro-F1 {base_combo['macro_f1']:.3f} ({time.time()-t0:.0f}s)")

## 4. Independent text set (optional): Symptom2Disease

In [ ]:
# Keyword rules map Symptom2Disease labels onto this project's condition names.
# Only labels with a clear equivalent are used; the printed table shows exactly what was mapped.
S2D_RULES = [
    ("psoriasis", "Psoriasis"), ("typhoid", "Typhoid"), ("chickenpox", "Chickenpox"), ("dengue", "Dengue"),
    ("commoncold", "CommonCold"), ("pneumonia", "Pneumonia"), ("hemorrhoid", "Hemorrhoids"),
    ("asthma", "Asthma"), ("hypertension", "Hypertension"), ("migraine", "Migraine"),
    ("cervicalspondylosis", "CervicalSpondylosis"), ("jaundice", "Jaundice"), ("malaria", "Malaria"),
    ("urinarytractinfection", "UrinaryInfection"), ("allergy", "Allergy"),
    ("gastroesophagealreflux", "GERD"), ("pepticulcer", "PepticUlcer"), ("diabetes", "DiabetesSymptoms"),
]

def norm(s): return "".join(ch for ch in str(s).lower() if ch.isalnum())

def map_s2d(label):
    n = norm(label)
    for key, target in S2D_RULES:
        if key in n and target in set(le.classes_):
            return target
    return None

df_ood = None
cands = [p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True) if "symptom2disease" in p.lower().replace("_", "")]
if cands:
    raw = pd.read_csv(cands[0])
    raw = raw[[c for c in raw.columns if c.lower() in ("label", "text")]].dropna()
    raw["mapped"] = raw["label"].map(map_s2d)
    df_ood = raw.dropna(subset=["mapped"]).rename(columns={"mapped": "target"}).reset_index(drop=True)
    df_ood["y"] = le.transform(df_ood["target"])
    print("file:", cands[0], "| rows:", len(raw), "| usable:", len(df_ood))
    print(raw.groupby("label")["mapped"].first().to_string())
else:
    print("Symptom2Disease not attached: skipping the independent test set (attach it for a stronger evaluation).")

if df_ood is not None:
    ood_vec = evaluate_probs(full_probs(clf, vec.transform(df_ood.text)), df_ood.y.values)
    print(f"TF-IDF + LR on independent text: acc {ood_vec['accuracy']:.3f}, macro-F1 {ood_vec['macro_f1']:.3f}")

## 5. Train and evaluate transformers

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

def load_tokenizer(hf_id):
    try:
        return AutoTokenizer.from_pretrained(hf_id)
    except Exception:
        return AutoTokenizer.from_pretrained(hf_id, use_fast=False)

def make_loader(tok, texts, labels, shuffle, bs):
    enc = tok(list(texts), truncation=True, max_length=MAX_LEN)
    items = [(enc["input_ids"][i], int(labels[i])) for i in range(len(labels))]
    def collate(batch):
        L = max(len(b[0]) for b in batch)
        ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        for i, (x, _) in enumerate(batch):
            ids[i, :len(x)] = torch.tensor(x); att[i, :len(x)] = 1
        return ids, att, torch.tensor([b[1] for b in batch])
    return DataLoader(items, batch_size=bs, shuffle=shuffle, collate_fn=collate)

def train_model(hf_id, seed, train_df):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    tok = load_tokenizer(hf_id)
    mdl = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=NC).to(device)
    loader = make_loader(tok, train_df.text.values, train_df.y.values, True, BATCH)
    total = len(loader) * EPOCHS
    opt = torch.optim.AdamW(mdl.parameters(), lr=LR, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * total), total)
    use_amp = device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    for ep in range(EPOCHS):
        mdl.train(); running = 0.0
        for step, (ids, att, y) in enumerate(loader):
            ids, att, y = ids.to(device), att.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = mdl(input_ids=ids, attention_mask=att, labels=y).loss
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            running += loss.item()
        print(f"   epoch {ep+1}/{EPOCHS}  loss {running/len(loader):.4f}")
    return mdl, tok

@torch.no_grad()
def predict_probs(mdl, tok, texts, bs=256):
    mdl.eval(); out = []
    loader = make_loader(tok, texts, np.zeros(len(texts), dtype=int), False, bs)
    for ids, att, _ in loader:
        logits = mdl(input_ids=ids.to(device), attention_mask=att.to(device)).logits
        out.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)

In [ ]:
runs = []                 # one row per (model, seed, split)
best = {"val_f1": -1}     # best run by VALIDATION macro-F1 (never the test set)
os.makedirs(BEST_DIR, exist_ok=True)
t_all = time.time()

for name, hf_id in MODELS.items():
    for seed in SEEDS:
        print(f"== {name} | seed {seed}")
        mdl, tok = train_model(hf_id, seed, train_combo)
        val = evaluate_probs(predict_probs(mdl, tok, val_combo.text.values), val_combo.y.values)
        p_test = predict_probs(mdl, tok, test_combo.text.values)
        r = evaluate_probs(p_test, test_combo.y.values)
        runs.append({"model": name, "seed": seed, "split": "combo", **r})
        p_ood = None
        if df_ood is not None:
            p_ood = predict_probs(mdl, tok, df_ood.text.values)
            runs.append({"model": name, "seed": seed, "split": "independent", **evaluate_probs(p_ood, df_ood.y.values)})
        print(f"   val F1 {val['macro_f1']:.3f} | held-out-combo acc {r['accuracy']:.3f} F1 {r['macro_f1']:.3f} | {time.time()-t_all:.0f}s elapsed")

        if val["macro_f1"] > best["val_f1"]:
            best = {"val_f1": val["macro_f1"], "model": name, "seed": seed, "hf_id": hf_id,
                    "p_test": p_test, "p_ood": p_ood}
            mdl.save_pretrained(BEST_DIR); tok.save_pretrained(BEST_DIR)
            joblib.dump(le, f"{BEST_DIR}/label_encoder.pkl")
        del mdl; torch.cuda.empty_cache()

# one run on the plain random split to show how much it inflates accuracy
first = list(MODELS)[0]
print(f"== {first} on the plain random split (reference)")
mdl, tok = train_model(MODELS[first], SEEDS[0], tr_r)
rr = evaluate_probs(predict_probs(mdl, tok, test_r.text.values), test_r.y.values)
runs.append({"model": first, "seed": SEEDS[0], "split": "random", **rr})
del mdl; torch.cuda.empty_cache()

runs_df = pd.DataFrame(runs)
runs_df.to_csv(f"{OUT}/runs.csv", index=False)
print("best by validation macro-F1:", best["model"], "seed", best["seed"], f"(val F1 {best['val_f1']:.3f})")

## 6. Results tables

In [ ]:
def ms(frame, col):
    return f"{frame[col].mean()*100:.1f} ± {frame[col].std(ddof=0)*100:.1f}" if col != "macro_f1" else f"{frame[col].mean():.3f} ± {frame[col].std(ddof=0):.3f}"

lines = []
for (m, s), g in runs_df.groupby(["model", "split"]):
    lines.append({"model": m, "split": s, "seeds": len(g), "accuracy %": ms(g, "accuracy"),
                  "macro F1": ms(g, "macro_f1"), "top-3 %": ms(g, "top3"), "ECE": f"{g['ece'].mean():.3f}"})
summary = pd.DataFrame(lines).sort_values(["split", "model"])
summary.to_csv(f"{OUT}/summary.csv", index=False)
print(summary.to_string(index=False))
print()
print(f"TF-IDF + LR baseline (held-out combos): acc {base_combo['accuracy']*100:.1f}%  macro-F1 {base_combo['macro_f1']:.3f}")

## 7. Calibration, selective prediction, confusions (best run)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

y_t = test_combo.y.values
p = best["p_test"]
sel = selective_accuracy(p, y_t)
ece_best = ece_score(p, y_t)
print(f"best model ECE on held-out combinations: {ece_best:.3f}")
for s in sel: print(f"  answers {s['coverage']*100:.0f}% of queries -> {s['accuracy']*100:.1f}% accurate")

# reliability diagram
conf, correct = p.max(1), (p.argmax(1) == y_t).astype(float)
edges = np.linspace(0, 1, 11); xs, ys = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (conf > lo) & (conf <= hi)
    if m.sum() >= 5: xs.append(conf[m].mean()); ys.append(correct[m].mean())
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
ax[0].plot([0, 1], [0, 1], "k--", alpha=.4); ax[0].plot(xs, ys, "o-")
ax[0].set_title(f"Reliability (ECE {ece_best:.3f})"); ax[0].set_xlabel("confidence"); ax[0].set_ylabel("accuracy")

covs = np.linspace(0.3, 1.0, 36)
order = np.argsort(-conf)
ax[1].plot(covs, [correct[order[:max(1, int(c*len(y_t)))]].mean() for c in covs])
ax[1].set_title("Selective prediction"); ax[1].set_xlabel("coverage (share of queries answered)"); ax[1].set_ylabel("accuracy")

# most confused pairs
pred = p.argmax(1); pairs = {}
for a, b in zip(y_t, pred):
    if a != b: pairs[(a, b)] = pairs.get((a, b), 0) + 1
top = sorted(pairs.items(), key=lambda kv: -kv[1])[:10]
ax[2].barh([f"{le.classes_[a]} -> {le.classes_[b]}" for (a, b), _ in top][::-1], [c for _, c in top][::-1])
ax[2].set_title("Most confused pairs"); ax[2].tick_params(axis="y", labelsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/calibration_and_confusions.png", dpi=150); plt.show()
confusions = [{"true": le.classes_[a], "predicted": le.classes_[b], "count": c} for (a, b), c in top]

## 8. Emergency rule engine
The emergency check is keyword and pattern based, and it runs **before** the model. This measures that rule engine alone on hand-written phrases that do not just repeat its keywords. Misses here are the most important finding: read them.

In [ ]:
from model.emergency_rules import check_emergency

EMERGENCY_POS = [
    "my father is clutching his chest and sweating badly", "she is not breathing and her lips are blue",
    "he collapsed and is not responding to us", "sudden numbness on one side of my face and slurred speech",
    "i am vomiting blood", "my child had a seizure just now", "bleeding heavily and it will not stop",
    "my tongue and throat are swelling after eating peanuts", "crushing chest pressure spreading to my left arm",
    "i cannot breathe properly at all", "severe headache and stiff neck with high fever", "he fainted and hit his head hard",
    "i think i am having a heart attack", "sudden weakness in my arm and my vision went blurry",
    "my baby is turning blue", "he is choking and cannot speak", "she has been unconscious for several minutes",
    "i have severe chest pain and shortness of breath", "my husband is having convulsions", "terrible abdominal pain and my belly is rock hard",
    "i swallowed a lot of pills and feel very drowsy", "i am coughing up blood", "sudden loss of vision in one eye",
    "my speech is slurred and my face is drooping", "he has a very high fever and is confused",
]
EMERGENCY_NEG = [
    "i have a mild headache since this morning", "runny nose and sneezing for two days", "my knee hurts after jogging",
    "slight sore throat and a little cough", "i feel tired and a bit dizzy when i stand up", "itchy rash on my arm for a week",
    "stomach ache after eating spicy food", "i have trouble sleeping lately", "mild back pain after sitting all day",
    "dry skin and itching on my legs", "my eyes are red and watery", "toothache since yesterday", "occasional heartburn after dinner",
    "i feel anxious before exams", "constipation for three days", "small cut on my finger that stopped bleeding",
    "low grade fever and body ache", "pain while urinating since yesterday", "blocked nose and mild cough",
    "muscle soreness after the gym", "i have acne on my face", "ear itching for a few days", "bloating after meals",
    "mild nausea in the morning", "i sprained my ankle yesterday and it is slightly swollen",
]
pos_hit = [bool(check_emergency(t)) for t in EMERGENCY_POS]
neg_hit = [bool(check_emergency(t)) for t in EMERGENCY_NEG]
tp, fp = sum(pos_hit), sum(neg_hit)
rules = {"recall": tp / len(EMERGENCY_POS), "precision": (tp / (tp + fp)) if (tp + fp) else None,
         "n_pos": len(EMERGENCY_POS), "n_neg": len(EMERGENCY_NEG), "false_alarms": fp}
print(f"emergency rules: recall {rules['recall']*100:.0f}% ({tp}/{len(EMERGENCY_POS)}), precision {0 if rules['precision'] is None else rules['precision']*100:.0f}%, false alarms {fp}")
print("MISSED emergencies:"); [print("  -", t) for t, h in zip(EMERGENCY_POS, pos_hit) if not h]
print("FALSE alarms:");       [print("  -", t) for t, h in zip(EMERGENCY_NEG, neg_hit) if h]

## 9. Latency (batch of 1)

In [ ]:
best_mdl = AutoModelForSequenceClassification.from_pretrained(BEST_DIR)
best_tok = load_tokenizer(BEST_DIR)
sample = "I have a runny nose, sneezing and a sore throat for two days"

def time_one(mdl, dev, n=60):
    mdl = mdl.to(dev).eval()
    enc = {k: v.to(dev) for k, v in best_tok(sample, return_tensors="pt", truncation=True, max_length=128).items()}
    with torch.no_grad():
        for _ in range(8): mdl(**enc)
        ts = []
        for _ in range(n):
            t = time.perf_counter(); mdl(**enc)
            if dev.type == "cuda": torch.cuda.synchronize()
            ts.append((time.perf_counter() - t) * 1000)
    return float(np.median(ts))

latency = {"cpu": time_one(copy.deepcopy(best_mdl), torch.device("cpu"))}
if device.type == "cuda":
    latency["gpu"] = time_one(copy.deepcopy(best_mdl), device)
print("median latency (ms):", {k: round(v, 1) for k, v in latency.items()})

## 10. Write the model card and results

In [ ]:
# model card numbers = mean over seeds of the architecture that won on validation
arch = best["model"]
def mean_row(split):
    g = runs_df[(runs_df.model == arch) & (runs_df.split == split)]
    return None if g.empty else {"accuracy": float(g.accuracy.mean()), "macro_f1": float(g.macro_f1.mean()),
                                 "top3": float(g.top3.mean()), "n": int(g.n.iloc[0])}
splits = []
rnd = runs_df[runs_df.split == "random"]
if not rnd.empty:
    r0 = rnd.iloc[0]
    splits.append({"name": f"Random split (symptom sets can repeat; {first}, 1 run)", "accuracy": float(r0.accuracy),
                   "macro_f1": float(r0.macro_f1), "top3": float(r0.top3), "n": int(r0.n)})
c = mean_row("combo")
splits.append({"name": f"Held-out symptom combinations ({arch}, mean of {len(SEEDS)} seeds)", **c})
o = mean_row("independent")
if o: splits.append({"name": f"Independent text set, Symptom2Disease ({arch}, mean of {len(SEEDS)} seeds)", **o})

baselines = [{"name": "TF-IDF + Logistic Regression (held-out combinations)",
              "accuracy": base_combo["accuracy"], "macro_f1": base_combo["macro_f1"]}]
for m in MODELS:
    if m != arch:
        g = runs_df[(runs_df.model == m) & (runs_df.split == "combo")]
        if not g.empty:
            baselines.append({"name": f"{m} (held-out combinations, mean of {len(SEEDS)} seeds)",
                              "accuracy": float(g.accuracy.mean()), "macro_f1": float(g.macro_f1.mean())})

metrics = {
    "evaluated_on": datetime.date.today().isoformat(),
    "n_classes": int(NC),
    "model": arch, "hf_base_model": MODELS[arch], "seeds": SEEDS,
    "splits": splits, "baselines": baselines,
    "calibration": {"ece": ece_best, "selective": sel},
    "emergency_rules": {k: rules[k] for k in ("recall", "precision", "n_pos", "n_neg")},
    "latency_ms": latency,
}

model_card = {
    "name": "MedAssist symptom triage", "version": "2.0",
    "summary": "A hybrid triage assistant: a fine-tuned transformer classifier suggests likely conditions, while rule engines handle emergencies, mental-health crises, severity and specialty routing.",
    "architecture": [
        "Input validation and sanitisation",
        "Intent detection (greeting, medicine query, crisis, symptoms)",
        "Rule-based emergency check (runs before the model)",
        f"{arch} classifier (top-3 conditions with confidence)",
        "Severity, specialty and care-advice rules",
    ],
    "training_data": {
        "source": "Synthetic symptom descriptions generated from symptom lists and sentence templates",
        "note": "No real patient data was used. Evaluation includes held-out symptom combinations and, when attached, an independent natural-language set.",
    },
    "evaluation_status": "measured",
    "metrics": metrics,
    "intended_use": "Educational demo of an AI triage workflow. It is not a medical device and is not clinically validated.",
    "limitations": [
        "Trained on synthetic text, so real-world accuracy is unknown until it is tested on independent clinical data.",
        "Covers a fixed list of conditions; anything outside it is mapped to the closest one.",
        "English only. Not tested for demographic fairness.",
        "Explanations show what the classifier reacted to, not medical causation.",
        f"The emergency rule engine caught {rules['recall']*100:.0f}% of the hand-written emergency phrases in testing, so it is not a safety guarantee.",
    ],
    "safety": [
        "Emergency patterns are detected by rules before the model runs.",
        "Low-confidence predictions return a prompt to see a clinician instead of a diagnosis.",
        "Every result carries a disclaimer to consult a qualified healthcare provider.",
    ],
}
json.dump(model_card, open(f"{OUT}/model-card.json", "w"), indent=2)
json.dump({"metrics": metrics, "best": {k: best[k] for k in ("model", "seed", "hf_id", "val_f1")},
           "random_split_test_overlap": float(leak), "confusions": confusions,
           "emergency_missed": [t for t, h in zip(EMERGENCY_POS, pos_hit) if not h],
           "emergency_false_alarms": [t for t, h in zip(EMERGENCY_NEG, neg_hit) if h]},
          open(f"{OUT}/eval_results.json", "w"), indent=2)
print(json.dumps(metrics, indent=2))

## 11. Optional: push the best model to Hugging Face
Set `PUSH_TO_HF = True` and add a Kaggle secret named `HF_TOKEN` (a write token). Push to a **new** repo name first.
Then set `HF_MODEL_ID=<your repo>` on the AI service (and rebuild or redeploy it) to switch over. Your existing model keeps working until you do.

In [ ]:
if PUSH_TO_HF:
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    api = HfApi(token=token)
    api.create_repo(HF_REPO, exist_ok=True, private=False)
    api.upload_folder(folder_path=BEST_DIR, repo_id=HF_REPO)
    print("uploaded. Now set HF_MODEL_ID =", HF_REPO, "on the AI service.")
else:
    print("PUSH_TO_HF is False: nothing uploaded. best_model/ is saved in the notebook output.")
shutil.make_archive(f"{WORK}/medassist_results", "zip", OUT)
print("results zipped ->", f"{WORK}/medassist_results.zip")

## What to copy back into the repo
1. `results/model-card.json` -> `backend/src/data/model-card.json` (the Model Card page shows it)
2. `results/*.png`, `summary.csv`, `runs.csv`, `eval_results.json` -> `ml/results/`
3. Save this notebook **with outputs** (Save Version -> Save & Run All) and commit it to `ml/`
4. If you pushed a model: set `HF_MODEL_ID` on the AI service